# RNN Classifier with LSTM Trained on Own Dataset (IMDB)

Example notebook showing how to use an own CSV text dataset for training a simple RNN for sentiment classification (here: a binary classification problem with two labels, positive and negative) using LSTM (Long Short Term Memory) cells.

In [24]:
import torch
import torch.nn.functional as F
import time
import random
import pandas as pd

torch.backends.cudnn.deterministic = True

## General Settings

In [25]:
RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)

VOCABULARY_SIZE = 20000
LEARNING_RATE = 0.005
BATCH_SIZE = 128
NUM_EPOCHS = 15
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 2

## Download Dataset

The following cells will download the IMDB movie review dataset (http://ai.stanford.edu/~amaas/data/sentiment/) for positive-negative sentiment classification in as CSV-formatted file:

In [26]:
!wget https://github.com/rasbt/python-machine-learning-book-3rd-edition/raw/master/ch08/movie_data.csv.gz

--2026-10-13 14:31:59--  https://github.com/rasbt/python-machine-learning-book-3rd-edition/raw/master/ch08/movie_data.csv.gz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch08/movie_data.csv.gz [following]
--2026-10-13 14:31:59--  https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch08/movie_data.csv.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26521894 (25M) [application/octet-stream]
Saving to: ‘movie_data.csv.gz’

movie_data.csv.gz   100%[===================>]  25.29M  --.-KB/s    in 0.1s    

2

In [27]:
!gunzip -f movie_data.csv.gz

Check that the dataset looks okay:

In [28]:
df = pd.read_csv('movie_data.csv')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [29]:
del df

## Prepare Dataset with Torchtext

In [30]:
!pip install spacy

Download English vocabulary via:
    
- `python -m spacy download en_core_web_sm`

Define the Label and Text field formatters:

In [31]:
### Defining the feature processing

import spacy


nlp = spacy.load('en_core_web_sm')  # same tokenizer as Field(tokenize='spacy')

def tokenize(text):
    return [tok.text for tok in nlp.tokenizer(text)]

Process the dataset:

In [32]:
df = pd.read_csv('movie_data.csv', dtype=str)

# each example is (list of tokens, label), like TabularDataset produced
dataset = [(tokenize(text), label) for text, label in zip(df.iloc[:, 0], df.iloc[:, 1])]
del df

## Split Dataset into Train/Validation/Test

Split the dataset into training, validation, and test partitions:

In [33]:
train_data, test_data = torch.utils.data.random_split(
    dataset, [0.8, 0.2],
    generator=torch.Generator().manual_seed(RANDOM_SEED))

print(f'Num Train: {len(train_data)}')
print(f'Num Test: {len(test_data)}')

Num Train: 40000
Num Test: 10000


In [34]:
train_data, valid_data = torch.utils.data.random_split(
    train_data, [0.85, 0.15],
    generator=torch.Generator().manual_seed(RANDOM_SEED))

print(f'Num Train: {len(train_data)}')
print(f'Num Validation: {len(valid_data)}')

Num Train: 34000
Num Validation: 6000


In [35]:
print(train_data[0])

(['The', 'GREAT', 'NEWS', 'is', 'that', 'this', 'film', 'is', 'now', 'AVAILABLE', 'on', 'DVD', 'from', 'http://treasureflix.com', 'for', 'all', 'those', 'who', 'wish', 'to', 'own', 'it', 'as', 'well', 'as', 'on', 'video', '.', 'This', 'is', 'good', 'news', 'as', 'it', 'is', 'one', 'of', 'my', 'favourite', 'films!<br', '/><br', '/>I', 'watched', 'this', 'film', 'for', 'the', 'first', 'time', 'in', 'the', '80s', 'and', 'it', 'is', 'compulsory', 'holiday', 'viewing', '.', 'Living', 'in', 'the', 'small', 'market', 'town', 'called', 'Tewkesbury', ',', 'picturesque', 'and', 'with', 'its', 'own', 'traditions', ',', 'of', 'reenactments', ',', 'and', 'traditions', 'we', 'are', 'also', 'a', 'cosy', 'tight', 'community', '.', 'We', 'are', 'now', 'also', 'faced', 'with', 'large', 'housing', 'developments', 'which', 'threaten', 'to', 'destroy', 'the', 'Community', 'and', 'you', 'can', 'see', 'why', 'I', 'love', 'this', 'film', 'First', 'of', 'all', '-', 'and', 'most', 'important', ',', 'there', 'ar

## Build Vocabulary

Build the vocabulary based on the top "VOCABULARY_SIZE" words:

In [36]:
from collections import Counter, defaultdict


class Vocab:
    """Replacement for torchtext's legacy Vocab (freqs, itos, stoi)."""

    def __init__(self, counter, max_size=None, specials=()):
        self.freqs = counter
        self.itos = list(specials) + [w for w, _ in counter.most_common(max_size)]
        # unknown words map to index 0 ('<unk>'), as in torchtext
        self.stoi = defaultdict(int, {w: i for i, w in enumerate(self.itos)})

    def __len__(self):
        return len(self.itos)


TEXT_VOCAB = Vocab(Counter(token for tokens, _ in train_data for token in tokens),
                   max_size=VOCABULARY_SIZE, specials=['<unk>', '<pad>'])
LABEL_VOCAB = Vocab(Counter(label for _, label in train_data))

print(f'Vocabulary size: {len(TEXT_VOCAB)}')
print(f'Number of classes: {len(LABEL_VOCAB)}')

Vocabulary size: 20002
Number of classes: 2


- 20,002 not 20,000 because of the `<unk>` and `<pad>` tokens
- PyTorch RNNs can deal with arbitrary lengths due to dynamic graphs, but padding is necessary for padding sequences to the same length in a given minibatch so we can store those in an array

**Look at most common words:**

In [37]:
print(TEXT_VOCAB.freqs.most_common(20))

[('the', 390911), (',', 368217), ('.', 318439), ('and', 210500), ('a', 210490), ('of', 194463), ('to', 179740), ('is', 146136), ('in', 118736), ('I', 105440), ('it', 103564), ('that', 94370), ('"', 85796), ("'s", 83204), ('this', 81393), ('-', 71103), ('/><br', 68674), ('was', 67783), ('movie', 57572), ('as', 57538)]


**Tokens corresponding to the first 10 indices (0, 1, ..., 9):**

In [38]:
print(TEXT_VOCAB.itos[:10]) # itos = integer-to-string

['<unk>', '<pad>', 'the', ',', '.', 'and', 'a', 'of', 'to', 'is']


**Converting a string to an integer:**

In [39]:
print(TEXT_VOCAB.stoi['the']) # stoi = string-to-integer

2


**Class labels:**

In [40]:
print(LABEL_VOCAB.stoi)

defaultdict(<class 'int'>, {'1': 0, '0': 1})


**Class label count:**

In [41]:
LABEL_VOCAB.freqs

Counter({'1': 17029, '0': 16971})

## Define Data Loaders

In [42]:
from collections import namedtuple
from torch.nn.utils.rnn import pad_sequence


class BucketSampler(torch.utils.data.Sampler):
    """Batches of similar-length examples (like BucketIterator)."""

    def __init__(self, data, batch_size, shuffle):
        order = sorted(range(len(data)), key=lambda i: len(data[i][0]))
        self.batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            random.shuffle(self.batches)
        return iter(self.batches)

    def __len__(self):
        return len(self.batches)


Batch = namedtuple('Batch', ['TEXT_COLUMN_NAME', 'LABEL_COLUMN_NAME'])

def collate(examples):
    texts = [torch.tensor([TEXT_VOCAB.stoi[t] for t in tokens]) for tokens, _ in examples]
    labels = torch.tensor([LABEL_VOCAB.stoi[label] for _, label in examples])
    text = pad_sequence(texts, padding_value=TEXT_VOCAB.stoi['<pad>'])  # [sentence length, batch size]
    return Batch(text.to(DEVICE), labels.to(DEVICE))


train_loader = torch.utils.data.DataLoader(
    train_data, batch_sampler=BucketSampler(train_data, BATCH_SIZE, shuffle=True), collate_fn=collate)
valid_loader = torch.utils.data.DataLoader(
    valid_data, batch_sampler=BucketSampler(valid_data, BATCH_SIZE, shuffle=False), collate_fn=collate)
test_loader = torch.utils.data.DataLoader(
    test_data, batch_sampler=BucketSampler(test_data, BATCH_SIZE, shuffle=False), collate_fn=collate)

Testing the iterators (note that the number of rows depends on the longest document in the respective batch):

In [43]:
print('Train')
for batch in train_loader:
    print(f'Text matrix size: {batch.TEXT_COLUMN_NAME.size()}')
    print(f'Target vector size: {batch.LABEL_COLUMN_NAME.size()}')
    break

print('\nValid:')
for batch in valid_loader:
    print(f'Text matrix size: {batch.TEXT_COLUMN_NAME.size()}')
    print(f'Target vector size: {batch.LABEL_COLUMN_NAME.size()}')
    break

print('\nTest:')
for batch in test_loader:
    print(f'Text matrix size: {batch.TEXT_COLUMN_NAME.size()}')
    print(f'Target vector size: {batch.LABEL_COLUMN_NAME.size()}')
    break

Train
Text matrix size: torch.Size([145, 128])
Target vector size: torch.Size([128])

Valid:
Text matrix size: torch.Size([56, 128])
Target vector size: torch.Size([128])

Test:
Text matrix size: torch.Size([51, 128])
Target vector size: torch.Size([128])


## Model

In [44]:
class RNN(torch.nn.Module):

    def __init__(self, input_dim, embedding_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = torch.nn.Embedding(input_dim, embedding_dim)
        #self.rnn = torch.nn.RNN(embedding_dim,
        #                        hidden_dim,
        #                        nonlinearity='relu')
        self.rnn = torch.nn.LSTM(embedding_dim,
                                 hidden_dim)

        self.fc = torch.nn.Linear(hidden_dim, output_dim)


    def forward(self, text):
        # text dim: [sentence length, batch size]

        embedded = self.embedding(text)
        # embedded dim: [sentence length, batch size, embedding dim]

        output, (hidden, cell) = self.rnn(embedded)
        # output dim: [sentence length, batch size, hidden dim]
        # hidden dim: [1, batch size, hidden dim]

        hidden.squeeze_(0)
        # hidden dim: [batch size, hidden dim]

        output = self.fc(hidden)
        return output

In [45]:
torch.manual_seed(RANDOM_SEED)
model = RNN(input_dim=len(TEXT_VOCAB),
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            output_dim=NUM_CLASSES # could use 1 for binary classification
)

model = model.to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

## Training

In [46]:
def compute_accuracy(model, data_loader, device):

    with torch.no_grad():

        correct_pred, num_examples = 0, 0

        for i, (features, targets) in enumerate(data_loader):

            features = features.to(device)
            targets = targets.float().to(device)

            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)

            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
    return correct_pred.float()/num_examples * 100

In [ ]:
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    for batch_idx, batch_data in enumerate(train_loader):

        text = batch_data.TEXT_COLUMN_NAME.to(DEVICE)
        labels = batch_data.LABEL_COLUMN_NAME.to(DEVICE)

        ### FORWARD AND BACK PROP
        logits = model(text)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()

        loss.backward()

        ### UPDATE MODEL PARAMETERS
        optimizer.step()

        ### LOGGING
        if not batch_idx % 50:
            print (f'Epoch: {epoch+1:03d}/{NUM_EPOCHS:03d} | '
                   f'Batch {batch_idx:03d}/{len(train_loader):03d} | '
                   f'Loss: {loss:.4f}')

    with torch.set_grad_enabled(False):
        print(f'training accuracy: '
              f'{compute_accuracy(model, train_loader, DEVICE):.2f}%'
              f'\nvalid accuracy: '
              f'{compute_accuracy(model, valid_loader, DEVICE):.2f}%')

    print(f'Time elapsed: {(time.time() - start_time)/60:.2f} min')

print(f'Total Training Time: {(time.time() - start_time)/60:.2f} min')
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

Epoch: 001/015 | Batch 000/266 | Loss: 0.7073
Epoch: 001/015 | Batch 050/266 | Loss: 0.6907
Epoch: 001/015 | Batch 100/266 | Loss: 0.6205
Epoch: 001/015 | Batch 150/266 | Loss: 0.5424
Epoch: 001/015 | Batch 200/266 | Loss: 0.4458
Epoch: 001/015 | Batch 250/266 | Loss: 0.3933
training accuracy: 84.65%
valid accuracy: 82.37%
Time elapsed: 0.29 min
Epoch: 002/015 | Batch 000/266 | Loss: 0.3728
Epoch: 002/015 | Batch 050/266 | Loss: 0.3677
Epoch: 002/015 | Batch 100/266 | Loss: 0.2439
Epoch: 002/015 | Batch 150/266 | Loss: 0.4208
Epoch: 002/015 | Batch 200/266 | Loss: 0.2147
Epoch: 002/015 | Batch 250/266 | Loss: 0.2960
training accuracy: 90.44%
valid accuracy: 86.05%
Time elapsed: 0.56 min
Epoch: 003/015 | Batch 000/266 | Loss: 0.1498
Epoch: 003/015 | Batch 050/266 | Loss: 0.2268
Epoch: 003/015 | Batch 100/266 | Loss: 0.1940
Epoch: 003/015 | Batch 150/266 | Loss: 0.2915
Epoch: 003/015 | Batch 200/266 | Loss: 0.1975
Epoch: 003/015 | Batch 250/266 | Loss: 0.2288
training accuracy: 95.89%
va

In [ ]:
import spacy


nlp = spacy.blank("en")

def predict_sentiment(model, sentence):

    model.eval()
    tokenized = [tok.text for tok in nlp.tokenizer(sentence)]
    indexed = [TEXT_VOCAB.stoi[t] for t in tokenized]
    length = [len(indexed)]
    tensor = torch.LongTensor(indexed).to(DEVICE)
    tensor = tensor.unsqueeze(1)
    length_tensor = torch.LongTensor(length)
    prediction = torch.nn.functional.softmax(model(tensor), dim=1)
    return prediction[0][0].item()

print('Probability positive:')
predict_sentiment(model, "This is such an awesome movie, I really love it!")

Probability positive:


0.9995642304420471

In [ ]:
print('Probability negative:')
1-predict_sentiment(model, "I really hate this movie. It is really bad and sucks!")

Probability negative:


0.9999993217898009